# 02 — Activity grain

Tag the grain of each activity, **per study**, as a first step towards matching activities to CDISC:

**named test / panel / procedure / instrument / sample collection / other / unsure**,
plus the structural roles **section header / specimen of parent / redacted** (excluded from the grain count, counted),
and for a named test the **kind of finding**: specimen / immunogenicity / measurement / instrument / pc / genomics / unsure.

Runs after consolidation (`01_batch.ipynb`), before the collection index and the activity inventory are rebuilt. Writes
`SoA2USDM/grain/<id>_activity_grain.json` per study (schema `soa2usdm-activity-grain` 1.0). The inventory is not changed.

Rules in `soa2usdm/activity_grain.py`; every row records the rule that fired and the lookup hit it cites. Kind of a rule-hit
named test: rule K1 through Domain_Coverage. Decisions (AI proposals, reviews) come from the hand-kept
`SoA2USDM/grain/<id>_activity_grain_decisions.json` where a study has one. Lookups are the public cdisc-for-ai files, from the
sibling checkout `../cdisc-for-ai` (or `SOA2USDM_CDISC_FOR_AI`); the run stops when those files have uncommitted changes.
Design, rule table and checks: `documents/activity-grain.md`.

## Configuration

In [ ]:
import sys
import json
from pathlib import Path

import pandas as pd

# Make the repo-root package importable (or `pip install -e '.[grain]'` from the repo root)
sys.path.insert(0, str(Path.cwd().parent))

from soa2usdm.activity_inventory import collect_rows
from soa2usdm.activity_grain import Lookups, write_study_grain, CDISC_FOR_AI

# === CONFIGURE ===
COLLECTION = 'cell_gene_therapy'
PROTOCOLS = None          # None = every study of the collection; or a list of protocol ids
SAMPLE_PER_RULE = 8

print('cdisc-for-ai:', CDISC_FOR_AI)

## Run

In [ ]:
lk = Lookups()
print('cdisc-for-ai commit:', lk.provenance['commit'])
all_rows = collect_rows(COLLECTION)
pids = PROTOCOLS or sorted({r['protocol_id'] for r in all_rows})
files = {pid: write_study_grain(COLLECTION, pid, lk, all_rows) for pid in pids}
grains = {pid: json.loads(f.read_text(encoding='utf-8')) for pid, f in files.items()}
labels = {(pid, l['phrase']): l for pid, g in grains.items() for l in g['labels']}
rows = pd.DataFrame([a for g in grains.values() for a in g['activities']])
for pid, g in grains.items():
    print(f"{pid}: {g['counts']['activities']} activities, {g['counts']['distinct_labels']} distinct labels, "
          f"decisions: {'yes' if g['inputs']['decisions'] else 'no (rules only)'}")

## Counts per study

Grain after decisions, and as the rules alone give it; roles; kind of finding of the named tests.

In [ ]:
def table(key):
    return pd.DataFrame({pid: g['counts'][key] for pid, g in grains.items()}).T.fillna(0).astype(int)

for key in ('by_grain', 'by_grain_rules_only', 'by_role', 'by_finding_kind', 'by_finding_kind_rules_only'):
    t = table(key)
    t.loc['total'] = t.sum()
    print(key)
    display(t)

## Counts per rule

In [ ]:
display(rows.groupby('rule_id').size().rename('rows').to_frame())

## Sample per rule

The first rows each rule fired on, with the hit it cites (strength and why weak) or the words that fired.

In [ ]:
def describe(r):
    l = labels.get((r['protocol_id'], r['phrase']), {})
    spec = r.get('specimen_hits')
    hits = l.get('hits') or (spec if isinstance(spec, list) else [])
    cite = '; '.join(f"{h.get('codelist', h['file'])}:{h['submission_value']} ({h['code']}, {h['field']})" for h in hits[:2])
    members = ', '.join(f"{m['text']}→{m['hits'][0]['submission_value']}" for m in l.get('members', []) if 'hits' in m)
    return pd.Series({
        'protocol': r['protocol_id'], 'activity': r['activity_name'],
        'grain / role': r['grain'] or r['role'], 'kind': r['finding_kind'], 'hit': cite,
        'strength': l.get('hit_strength'), 'weak because': ', '.join(l.get('weak_reasons', [])),
        'words': '; '.join(f'{k}: {v}' for k, v in (l.get('words') or {}).items()),
        'members': members, 'flags': ', '.join(r['flags'] + l.get('flags', [])),
    })

with pd.option_context('display.max_colwidth', 80, 'display.max_rows', 500):
    for rule, grp in rows.groupby('rule_id', sort=True):
        print(f'{rule} — {len(grp)} rows')
        display(grp.head(SAMPLE_PER_RULE).apply(describe, axis=1))

## Named tests and their kind

Who set grain and kind (rule K1, or the source of a decision), the K1 domains, and the decision's rationale where there is one.

In [ ]:
def named(r):
    l = labels[(r['protocol_id'], r['phrase'])]
    d = l.get('decision', {})
    return pd.Series({'protocol': r['protocol_id'], 'activity': r['activity_name'], 'kind': r['finding_kind'],
                      'source': r['source'], 'K1 domains': ', '.join(l['kind_domains']),
                      'rationale': d.get('rationale', '')})

with pd.option_context('display.max_colwidth', 140, 'display.max_rows', 500):
    nt = rows[rows['grain'] == 'named test']
    display(nt.apply(named, axis=1).sort_values(['kind', 'protocol']))

## Unsure — the remainder

Distinct labels of each study that neither the rules nor a decision resolved, with the number of rows. In a study graded by
rules only, unsure means not yet resolved.

In [ ]:
unsure = rows[rows['grain'] == 'unsure']
with pd.option_context('display.max_colwidth', 100, 'display.max_rows', 500):
    display(unsure.groupby(['protocol_id', 'phrase']).agg(rows=('xact_id', 'size'),
                                                          activity=('activity_name', 'first'),
                                                          rule=('rule_id', 'first'))
            .sort_values('rows', ascending=False))

## Weak hits to review

In [ ]:
weak = [(pid, p, l['spellings'][0], l['grain'], l['rule_id'], l['hits'][0]['submission_value'], ', '.join(l['weak_reasons']))
        for (pid, p), l in labels.items() if l.get('hit_strength') == 'weak']
with pd.option_context('display.max_colwidth', 80, 'display.max_rows', 500):
    display(pd.DataFrame(weak, columns=['protocol', 'phrase', 'activity', 'grain', 'rule', 'hit', 'weak because']))